<a href="https://colab.research.google.com/github/mahadikprasad15/ARENA/blob/main/Attention_and_Prompted_probes_generalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformer_lens

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 46.9 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=eae1bbd46f40576c8668117e91e76ff9c1f5f792ef06790c948ecf52ea161af9
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: beartype
    Found existing installa

# Setup files

Downloading necessary modules

In [1]:
import transformer_lens
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


import numpy as np
import pandas as pd
import os
import json
import requests
from pathlib import Path
from typing import List, Dict
from typing import Optional, Literal
from collections import Counter
import random

import plotly.express as px
import matplotlib

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

## Downloading the Model

In [2]:
model = transformer_lens.HookedTransformer.from_pretrained("Qwen/Qwen2.5-0.5B")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer


## Downloading the Train and Test datasets

In [3]:
DATA_DIR = Path("data/high_stakes")
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_url = "https://pub-fd16e959a4f14ca48765b437c9425ba6.r2.dev/training/prompts_4x/train.jsonl"
train_path = DATA_DIR / "train.jsonl"

response = requests.get(train_url)
response.raise_for_status()

train_path.write_bytes(response.content)
print("Saved train data to", train_path)


MT_url = "https://pub-fd16e959a4f14ca48765b437c9425ba6.r2.dev/evals/dev/mt_balanced_apr_30.jsonl"
MT_dev_path = DATA_DIR / "MT_dev.jsonl"


response = requests.get(MT_url)
response.raise_for_status()
MT_dev_path.write_bytes(response.content)

print('Saved test data to', MT_dev_path)


Saved train data to data/high_stakes/train.jsonl
Saved test data to data/high_stakes/MT_dev.jsonl


In [4]:
def load_jsonl(path) -> List[Dict]:
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data.append(json.loads(line))
    return data

def label_to_int(x: str) -> int:
    if x == "high-stakes":
        return 1
    elif x == "low-stakes":
        return 0
    else:
        raise ValueError(f"Unexpected label: {x!r}")


def normalize_inputs(inputs_field: str) -> str:
    s = inputs_field.strip()


    if s.startswith('[') and '"role"' in s:
        try:
            messages = json.loads(s)
            parts = [f"{m['role']}: {m['content']}" for m in messages]
            return "\n".join(parts)
        except json.JSONDecodeError:

            return inputs_field
    else:

        return inputs_field


In [5]:
train_rows = load_jsonl("data/high_stakes/train.jsonl")
dev_rows   = load_jsonl("data/high_stakes/MT_dev.jsonl")
len(train_rows), len(dev_rows)

(8000, 278)

In [6]:
train_dataset = [{'text': normalize_inputs(row['inputs']), 'label': label_to_int(row['labels'])} for row in train_rows]
test_dataset = [{'text': normalize_inputs(row['inputs']), 'label': label_to_int(row['labels'])} for row in dev_rows]

train_texts = [train['text'] for train in train_dataset]
train_labels= [train['label'] for train in train_dataset]

test_texts = [test['text'] for test in test_dataset]
test_labels= [test['label'] for test in test_dataset]


combined = list(zip(train_texts, train_labels))

random.shuffle(combined)

train_texts, train_labels = zip(*combined)
train_texts = list(train_texts)
train_labels = list(train_labels)

In [7]:
def create_dataloaders(
    activations: np.ndarray,
    labels: List[int],
    batch_size: int = 32,
    train_split: float = 0.8
):
    """Create train/val dataloaders from activations and labels"""

    # Convert to tensors
    X = torch.FloatTensor(activations)
    y = torch.FloatTensor(labels)

    # Create dataset
    dataset = TensorDataset(X, y)

    # Split train/val if needed
    if train_split < 1.0:
        train_size = int(train_split * len(dataset))
        val_size = len(dataset) - train_size
        train_dataset, val_dataset = torch.utils.data.random_split(
            dataset, [train_size, val_size]
        )

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        return train_loader, val_loader
    else:
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        return loader


In [8]:
def get_activations(texts, model, layer_idx=-1, batch_size=8, pooling='last', pad_all=True):
    """
    Extract activations with different pooling strategies.

    Args:
        pad_all: If True and pooling='all', pad sequences to same length
    """
    model.eval()
    all_activations = []

    if layer_idx < 0:
      hook_name = f'blocks.{model.cfg.n_layers+layer_idx}.hook_resid_post'
    else:
      hook_name = f'blocks.{layer_idx}.hook_resid_post'

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]

        for text in batch_texts:
            captured = []

            def hook_fn(activation, hook):
                captured.append(activation.clone().cpu())

            with torch.no_grad():
                model.run_with_hooks(
                    text,
                    fwd_hooks=[(hook_name, hook_fn)]
                )

            hidden_states = captured[0][0]  # [seq_len, d_model]

            if pooling == 'last':
                act = hidden_states[-1, :]
            elif pooling == 'mean':
                act = hidden_states.mean(dim=0)
            elif pooling == 'first':
                act = hidden_states[0, :]
            elif pooling == 'all':
                act = hidden_states  # [seq_len, d_model]

            all_activations.append(act)
            del captured, hidden_states, act

        torch.cuda.empty_cache()

    # Concatenate based on pooling
    if pooling == 'all':
        if pad_all:
            # Pad to same length
            from torch.nn.utils.rnn import pad_sequence
            padded = pad_sequence(all_activations, batch_first=True)
            return padded.numpy()  # [num_texts, max_seq_len, d_model]
        else:
            return all_activations  # List of varying length tensors
    else:
        return torch.stack(all_activations, dim=0).numpy()

In [51]:
class LinearProbe(nn.Module):


    def __init__(self, d_model: int, n_classes: int = 1):
        super().__init__()

        self.linear = nn.Linear(d_model, n_classes, bias = True)

    def forward(self, activations: torch.Tensor) -> torch.Tensor:
        """
        Args:
            activations: shape [batch, seq_len, d_model] OR [batch, d_model]
        Returns:
            logits: shape [batch, n_classes]
        """
        return self.linear(activations)



class AttentionProbe(nn.Module):
    def __init__(self, d_model: int):
        """
        d_model: hidden size of the LM layer you’re probing.
        """
        super().__init__()

        self.q = nn.Linear(d_model, 1, bias=True)
        self.classifier = nn.Linear(d_model, 1, bias=True)


    def forward(self, x):
            """
            x: [batch, seq_len, d_model]
            Returns: logits [batch, 1]
            """
            scores = self.q(x).squeeze(-1)      # [B, T]
            attn = F.softmax(scores, dim=-1)               # [B, T]
            pooled = (attn.unsqueeze(-1) * x).sum(dim=1)   # [B, d_model]
            logits = self.classifier(pooled)               # [B, 1]
            return logits


In [10]:
class ProbeTrainer:
    def __init__(
        self,
        probe: nn.Module,
        learning_rate: float = 1e-3,
        weight_decay: float = 0.01,
        device: str = "cuda" if torch.cuda.is_available() else "cpu"
    ):
        self.probe = probe.to(device)
        self.device = device
        self.optimizer = torch.optim.Adam(probe.parameters(), lr=learning_rate, weight_decay=weight_decay)
        self.criterion = torch.nn.BCEWithLogitsLoss()

    def fit(
        self,
        train_activations: np.ndarray,
        train_labels: List[int],
        epochs: int = 100,
        batch_size: int = 32,
        patience: int = 10,
        train_split: float = 0.8  # Add this parameter for flexibility
    ):

        # Use create_dataloaders to handle train/val split
        train_loader, val_loader = create_dataloaders(
            train_activations,
            train_labels,
            batch_size=batch_size,
            train_split=train_split
        )

        best_val_loss = float('inf')
        patience_counter = 0

        for epoch in range(epochs):

            # Training
            self.probe.train()
            total_loss = 0
            num_batches = 0

            for x, y in train_loader:
                x = x.to(self.device)
                y = y.to(self.device).unsqueeze(1)

                self.optimizer.zero_grad()
                output = self.probe(x)
                loss = self.criterion(output, y)
                loss.backward()
                self.optimizer.step()

                total_loss += loss.item()
                num_batches += 1

            avg_train_loss = total_loss / num_batches

            # Validation
            self.probe.eval()
            val_loss = 0
            num_val_batches = 0

            with torch.no_grad():
                for x, y in val_loader:
                    x = x.to(self.device)
                    y = y.to(self.device).unsqueeze(1)
                    output = self.probe(x)
                    loss = self.criterion(output, y)
                    val_loss += loss.item()
                    num_val_batches += 1

            avg_val_loss = val_loss / num_val_batches

            # Early stopping
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break

            if epoch % 10 == 0:
                print(f"Epoch {epoch}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    def evaluate(
        self,
        activations: np.ndarray,
        labels: List[int],
        batch_size: int = 32
    ) -> dict:
        """Evaluate probe on a dataset"""
        from sklearn.metrics import roc_auc_score

        loader = create_dataloaders(
            activations,
            labels,
            batch_size=batch_size,
            train_split=1.0
        )

        self.probe.eval()
        all_probs = []  # For AUROC - keep probabilities
        all_preds = []  # For binary metrics
        all_labels = []
        total_loss = 0

        with torch.no_grad():
            for x, y in loader:
                x = x.to(self.device)
                y = y.to(self.device)

                logits = self.probe(x)

                # Handle shape issues: ensure 1D for loss
                logits_squeezed = logits.squeeze()
                y_squeezed = y.squeeze()

                loss = self.criterion(logits_squeezed, y_squeezed)
                total_loss += loss.item()

                # Get probabilities (before thresholding) for AUROC
                probs = torch.sigmoid(logits_squeezed)

                # Binary predictions (threshold at 0.5)
                preds = (probs > 0.5).float()

                all_probs.append(probs.detach().cpu())
                all_preds.append(preds.detach().cpu())
                all_labels.append(y_squeezed.detach().cpu())

        # Concatenate all batches and convert to numpy with proper shapes
        probs = torch.cat(all_probs).numpy().flatten()
        preds = torch.cat(all_preds).numpy().flatten().astype(np.int32)
        labels_array = torch.cat(all_labels).numpy().flatten().astype(np.int32)

        # Ensure no NaN/Inf issues
        probs = np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0)

        # Compute metrics
        accuracy = accuracy_score(labels_array, preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels_array, preds, average='binary', zero_division=0
        )
        auroc = roc_auc_score(labels_array, probs)

        return {
            'accuracy': float(accuracy),
            'precision': float(precision),
            'recall': float(recall),
            'f1': float(f1),
            'auroc': float(auroc),
            'loss': float(total_loss / len(loader))
        }

    def predict(
        self,
        activations: np.ndarray,
        batch_size: int = 32
    ) -> np.ndarray:
        """Get predictions for activations"""
        loader = create_dataloaders(
            activations,
            np.zeros(len(activations)),  # Dummy labels (not used)
            batch_size=batch_size,
            train_split=1.0
        )

        self.probe.eval()
        all_preds = []

        with torch.no_grad():
            for x, _ in loader:
                x = x.to(self.device)
                logits = self.probe(x)
                preds = (logits.sigmoid() > 0.5).float().squeeze()
                all_preds.append(preds.cpu().numpy())

        return np.concatenate(all_preds)

In [ ]:
train_activations = get_activations(train_texts, model, layer_idx=-1, batch_size=8, pooling='mean', pad_all=True)

In [ ]:
probe = LinearProbe(model.cfg.d_model)
probe_trainer = ProbeTrainer(probe)
probe_trainer.fit(train_activations, train_labels[:1000])

In [11]:
probe = LinearProbe(model.cfg.d_model)

In [12]:
def train_probes_all_layers(train_texts, train_labels, model, pooling = 'mean'):

  probes = {}

  for layer in tqdm(range(model.cfg.n_layers)):
    print(f'Training Probe for layer {layer}')
    train_activations = get_activations(train_texts, model, layer_idx = layer, batch_size=8, pooling= pooling, pad_all=True)
    probe = LinearProbe(model.cfg.d_model)
    probe_trainer = ProbeTrainer(probe)
    probe_trainer.fit(train_activations, train_labels)

    result = probe_trainer.evaluate(train_activations, train_labels)
    probes[layer] = {'result':result,
                     'probe':probe}

    del train_activations
    torch.cuda.empty_cache()
  return probes

In [16]:
def test_all_probes(probes, test_texts, test_labels, model, pooling = 'mean'):
  test_results = {}
  for layer, probe in probes.items():

    test_activations = get_activations(test_texts, model, layer_idx = layer, batch_size=8, pooling= pooling, pad_all=True)

    probe_layer = probe['probe']
    probe_layer.eval()
    probe_trainer = ProbeTrainer(probe_layer)
    probe_evaluation_result = probe_trainer.evaluate(test_activations, test_labels)

    print(f'Evaluation_result for layer {layer} \n {probe_evaluation_result}')
    print('\n')

    test_results[layer] = probe_evaluation_result

    del test_activations
    torch.cuda.empty_cache()

  return test_results


In [14]:
probes_all_layers = train_probes_all_layers(train_texts[:500], train_labels[:500], model, pooling = 'mean')

  0%|          | 0/24 [00:00<?, ?it/s]

Training Probe for layer 0



100%|██████████| 63/63 [00:45<00:00,  1.40it/s]


Epoch 0/100 | Train Loss: 0.6907 | Val Loss: 0.6932
Epoch 10/100 | Train Loss: 0.6460 | Val Loss: 0.6514
Epoch 20/100 | Train Loss: 0.6204 | Val Loss: 0.6346
Epoch 30/100 | Train Loss: 0.6039 | Val Loss: 0.6240
Epoch 40/100 | Train Loss: 0.5933 | Val Loss: 0.6203
Epoch 50/100 | Train Loss: 0.5867 | Val Loss: 0.6107
Epoch 60/100 | Train Loss: 0.5811 | Val Loss: 0.6068
Epoch 70/100 | Train Loss: 0.5753 | Val Loss: 0.6012


  4%|▍         | 1/24 [00:49<19:06, 49.85s/it]

...

Epoch 40/100 | Train Loss: 0.1847 | Val Loss: 0.3822
Early stopping at epoch 43
Training Probe for layer 23



100%|██████████| 63/63 [00:34<00:00,  1.80it/s]


Epoch 0/100 | Train Loss: 0.6080 | Val Loss: 0.5851
Epoch 10/100 | Train Loss: 0.2717 | Val Loss: 0.3397
Epoch 20/100 | Train Loss: 0.2035 | Val Loss: 0.3096
Epoch 30/100 | Train Loss: 0.1791 | Val Loss: 0.2936
Epoch 40/100 | Train Loss: 0.1521 | Val Loss: 0.2800
Epoch 50/100 | Train Loss: 0.1345 | Val Loss: 0.2743
Epoch 60/100 | Train Loss: 0.1242 | Val Loss: 0.2712


100%|██████████| 24/24 [16:32<00:00, 41.36s/it]

Epoch 70/100 | Train Loss: 0.1209 | Val Loss: 0.2696
Early stopping at epoch 74


In [17]:
evaluation_for_all_layers = test_all_probes(probes_all_layers, test_texts[:500], test_labels[:500], model, pooling = 'mean')


100%|██████████| 35/35 [01:17<00:00,  2.22s/it]


Evaluation_result for layer 0 
 {'accuracy': 0.6438848920863309, 'precision': 0.5869565217391305, 'recall': 0.9712230215827338, 'f1': 0.7317073170731707, 'auroc': 0.7573106982040267, 'loss': 0.6723393334282769}




100%|██████████| 35/35 [01:20<00:00,  2.30s/it]


Evaluation_result for layer 1 
 {'accuracy': 0.6330935251798561, 'precision': 0.5787234042553191, 'recall': 0.9784172661870504, 'f1': 0.7272727272727273, 'auroc': 0.7523420112830599, 'loss': 0.6645939151446024}




100%|██████████| 35/35 [01:19<00:00,  2.27s/it]


...

100%|██████████| 35/35 [01:18<00:00,  2.25s/it]


Evaluation_result for layer 22 
 {'accuracy': 0.6834532374100719, 'precision': 0.6363636363636364, 'recall': 0.8561151079136691, 'f1': 0.7300613496932515, 'auroc': 0.7605713989959111, 'loss': 0.7209557625982497}




100%|██████████| 35/35 [01:18<00:00,  2.25s/it]

Evaluation_result for layer 23 
 {'accuracy': 0.7086330935251799, 'precision': 0.6479591836734694, 'recall': 0.9136690647482014, 'f1': 0.7582089552238805, 'auroc': 0.7886755343926298, 'loss': 0.8407827085918851}




{0: {'accuracy': 0.6438848920863309,
  'precision': 0.5869565217391305,
  'recall': 0.9712230215827338,
  'f1': 0.7317073170731707,
  'auroc': 0.7573106982040267,
  'loss': 0.6723393334282769},
 1: {'accuracy': 0.6330935251798561,
  'precision': 0.5787234042553191,
  'recall': 0.9784172661870504,
  'f1': 0.7272727272727273,
  'auroc': 0.7523420112830599,
  'loss': 0.6645939151446024},
 2: {'accuracy': 0.5035971223021583,
  'precision': 0.5018050541516246,
  'recall': 1.0,
  'f1': 0.6682692307692307,
  'auroc': 0.7357797215465038,
  'loss': 0.6834614475568136},
 3: {'accuracy': 0.6546762589928058,
  'precision': 0.5955555555555555,
  'recall': 0.9640287769784173,
  'f1': 0.7362637362637363,
  'auroc': 0.79374773562445,
  'loss': 0.6449322170681424},
 4: {'accuracy': 0.5719424460431655,
  'precision': 0.5387596899224806,
  'recall': 1.0,
  'f1': 0.7002518891687658,
  'auroc': 0.7965943791729206,
  'loss': 0.6540350516637167},
 5: {'accuracy': 0.5755395683453237,
  'precision': 0.54085603

In [29]:
pd.DataFrame(evaluation_for_all_layers).transpose()


,accuracy,precision,recall,f1,auroc,loss
0,0.643885,0.586957,0.971223,0.731707,0.757311,0.672339
1,0.633094,0.578723,0.978417,0.727273,0.752342,0.664594
2,0.503597,0.501805,1.000000,0.668269,0.735780,0.683461
3,0.654676,0.595556,0.964029,0.736264,0.793748,0.644932
4,0.571942,0.538760,1.000000,0.700252,0.796594,0.654035
5,0.575540,0.540856,1.000000,0.702020,0.848869,0.630431
6,0.651079,0.588983,1.000000,0.741333,0.831168,0.619861
7,0.737410,0.673684,0.920863,0.778116,0.826148,0.585880
8,0.683453,0.615385,0.978417,0.755556,0.852647,0.588706
9,0.769784,0.692308,0.971223,0.808383,0.881217,0.558290


In [2]:
probes_all_layers_results = {key : value['result'] for key, value in probes_all_layers.items()}
pd.DataFrame(probes_all_layers_results).transpose()

NameError: name 'probes_all_layers' is not defined

In [1]:
layer = 10
train_acts_all = get_activations(
    train_texts[:1000],
    model,
    layer_idx=layer,
    batch_size=8,
    pooling='all',
    pad_all=True
)

attn_probe = AttentionProbe(model.cfg.d_model)
attn_trainer = ProbeTrainer(attn_probe)
attn_trainer.fit(train_acts_all, train_labels[:1000], train_split = 0.8)

del train_acts_all
gc.collect()
torch.cuda.empty_cache()



test_acts = get_activations(
    test_texts[:500],
    model,
    layer_idx=layer,
    batch_size=8,
    pooling='all',
    pad_all=True
)
attn_metrics = attn_trainer.evaluate(test_acts, test_labels)
print("Attention probe:", attn_metrics)





NameError: name 'get_activations' is not defined